# Fast HMM on Google Speech Commands (Tiny Subset)

This notebook downloads **Speech Commands v0.02**, extracts a **small subset** of labels/utterances, and trains **one diagonal-Gaussian HMM per word** for a quick speech-recognition demo.

### Built-in speedups
- Coarse framing and **frame subsampling** with **max-frames cap**
- **8 MFCCs** at **8 kHz**
- **Left–right** (banded) transitions
- **Diagonal** covariances with **variance floor**
- **Mini-batch EM** with **few iterations**
- Fixed broadcasting via `np.logaddexp.reduce`


## 0. Config & Imports

In [1]:
import numpy as np, os, sys, tarfile, hashlib, urllib.request, shutil, glob, traceback
import librosa
from typing import List, Dict
np.set_printoptions(precision=3, suppress=True)

# --- DOWNLOAD CONFIG ---
URL = "http://download.tensorflow.org/data/speech_commands_v0.02.tar.gz"
ARCHIVE = "/mnt/data/speech_commands_v0.02.tar.gz"
EXTRACT_DIR = "/mnt/data/speech_commands_v0.02"

# --- SUBSET CONFIG ---
USE_LABELS = ['yes','no','up','down']   # choose a few labels for the demo
MAX_UTTS_PER_LABEL = 12                 # cap utterances per label

# --- FEATURE CONFIG (fast) ---
SR        = 8000      # downsample to 8 kHz
N_MFCC    = 8         # fewer coefficients
HOP       = 320       # 40 ms hop at 8 kHz
N_FFT     = 400       # 50 ms window
FRAME_SUBSAMPLE = 2   # keep every 2nd frame
MAX_FRAMES = 80

# --- HMM CONFIG (fast) ---
K = 3
ITERS = 10
TOL = 1e-3
VAR_FLOOR = 1e-3
LEFT_RIGHT = True
USE_MINIBATCH = True
BATCH_FRACTION = 0.5

rng = np.random.default_rng(0)


## 1. Download & Extract Speech Commands v0.02 (once)

In [2]:
def download_if_needed(url: str, path: str):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    if os.path.isfile(path) and os.path.getsize(path) > 0:
        print("Archive already present:", path)
        return path
    print("Downloading:", url)
    with urllib.request.urlopen(url) as r, open(path, 'wb') as f:
        shutil.copyfileobj(r, f)
    print("Saved:", path)
    return path

def extract_if_needed(archive: str, dest_dir: str):
    if os.path.isdir(dest_dir) and len(os.listdir(dest_dir)) > 0:
        print("Already extracted:", dest_dir)
        return dest_dir
    print("Extracting to:", dest_dir)
    os.makedirs(dest_dir, exist_ok=True)
    with tarfile.open(archive, 'r:gz') as tar:
        tar.extractall(dest_dir)
    # The archive contains folders directly; some dumps add a top-level. Handle both.
    return dest_dir

try:
    download_if_needed(URL, ARCHIVE)
    extract_if_needed(ARCHIVE, EXTRACT_DIR)
    print("Ready.")
except Exception as e:
    print(f"[download/extract] {e}")
    traceback.print_exc()


Downloading: http://download.tensorflow.org/data/speech_commands_v0.02.tar.gz
Saved: /mnt/data/speech_commands_v0.02.tar.gz
Extracting to: /mnt/data/speech_commands_v0.02


/tmp/ipython-input-2568485336.py:19: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(dest_dir)


Ready.


## 2. Build a Tiny Subset & Extract MFCCs

In [3]:
def gather_label_wavs(root: str, label: str):
    # The dataset unpacks folders like 'yes/', 'no/', etc.
    pat = os.path.join(root, label, '*.wav')
    return sorted(glob.glob(pat))

def load_mfcc(path, sr=SR, n_mfcc=N_MFCC, hop_length=HOP, n_fft=N_FFT):
    y, s = librosa.load(path, sr=sr)
    mfcc = librosa.feature.mfcc(y=y, sr=s, n_mfcc=n_mfcc, hop_length=hop_length, n_fft=n_fft)
    X = mfcc.T
    if FRAME_SUBSAMPLE > 1:
        X = X[::FRAME_SUBSAMPLE]
    if X.shape[0] > MAX_FRAMES:
        X = X[:MAX_FRAMES]
    return X.astype(np.float32)

def build_subset(root: str, labels: List[str], max_utts: int):
    Xs, ys, names = [], [], []
    for li, lab in enumerate(labels):
        wavs = gather_label_wavs(root, lab)
        if len(wavs) == 0:
            print(f"[warn] No WAVs for label '{lab}' under {root}")
            continue
        rng.shuffle(wavs)
        wavs = wavs[:max_utts]
        for w in wavs:
            try:
                X = load_mfcc(w)
                if X.shape[0] >= 3:
                    Xs.append(X)
                    ys.append(li)
                    names.append((lab, os.path.basename(w)))
            except Exception as e:
                print(f"[load_mfcc] {w}: {e}")
                traceback.print_exc()
    if len(Xs) == 0:
        raise RuntimeError("No usable sequences found in the subset.")
    return Xs, np.array(ys), names

ROOT = EXTRACT_DIR
X, y, NAMES = build_subset(ROOT, USE_LABELS, MAX_UTTS_PER_LABEL)
print(f"Subset: {len(X)} utterances across {len(USE_LABELS)} labels -> {USE_LABELS}")
print("Example sequence shape:", X[0].shape)


Subset: 48 utterances across 4 labels -> ['yes', 'no', 'up', 'down']
Example sequence shape: (13, 8)


## 3. Fast HMM (Diagonal Gaussians, Left–Right, Mini-batch)

In [4]:
def logsumexp_np(a, axis=None):
    return np.logaddexp.reduce(a, axis=axis)

def normalize_rows(M):
    s = M.sum(axis=1, keepdims=True)
    s[s==0] = 1.0
    return M / s

def banded_left_right_A(K:int, p_stay:float=0.7):
    A = np.zeros((K, K), dtype=float)
    for i in range(K):
        A[i, i] = p_stay
        if i < K-1:
            A[i, i+1] = 1.0 - p_stay
    return normalize_rows(A)

def log_diag_gauss_pdf(X:np.ndarray, means:np.ndarray, vars_:np.ndarray):
    T, D = X.shape
    K = means.shape[0]
    log_norm = -0.5*(D*np.log(2*np.pi) + np.sum(np.log(vars_), axis=1))  # (K,)
    X_exp = X[:, None, :]
    num = (X_exp - means[None, :, :])**2
    denom = vars_[None, :, :]
    mahal = -0.5 * np.sum(num / denom, axis=2)
    return mahal + log_norm[None, :]

def forward_backward_log(log_pi, log_A, log_B):
    T, K = log_B.shape
    alpha = np.empty((T, K))
    alpha[0] = log_pi + log_B[0]
    for t in range(1, T):
        tmp = alpha[t-1][:, None] + log_A  # (K,K)
        alpha[t] = log_B[t] + logsumexp_np(tmp, axis=0)
    logZ = logsumexp_np(alpha[-1], axis=0)
    beta = np.empty((T, K)); beta[-1] = 0.0
    for t in range(T-2, -1, -1):
        tmp = log_A + (log_B[t+1] + beta[t+1])[None, :]
        beta[t] = logsumexp_np(tmp, axis=1)
    gamma = alpha + beta
    gamma = gamma - logsumexp_np(gamma, axis=1)[:, None]
    gamma = np.exp(gamma)
    xi = np.empty((T-1, K, K))
    for t in range(T-1):
        M = alpha[t][:, None] + log_A + log_B[t+1][None, :] + beta[t+1][None, :]
        M = M - logsumexp_np(M)
        xi[t] = np.exp(M)
    return logZ, gamma, xi

def init_params(K:int, D:int, left_right=True, rng=None):
    rng = rng or np.random.default_rng(0)
    pi = np.zeros(K)
    pi[0] = 1.0 if left_right else 1.0/K
    A = banded_left_right_A(K) if left_right else normalize_rows(0.7*np.eye(K) + 0.3*rng.random((K,K)))
    means = rng.normal(0, 1, size=(K, D))
    vars_ = np.ones((K, D))*0.5
    return dict(pi=pi, A=A, means=means, vars_=vars_)

def baum_welch_diagonal_gaussian(seqs: List[np.ndarray], K:int, iters:int=10, tol:float=1e-3,
                                 rng=None, verbose:bool=False, init=None,
                                 var_floor:float=1e-3, left_right:bool=True):
    rng = rng or np.random.default_rng(0)
    D = seqs[0].shape[1]
    params = init_params(K, D, left_right, rng) if init is None else init.copy()
    pi, A, means, vars_ = params['pi'], params['A'], params['means'], params['vars_']
    prev_ll = -np.inf
    for it in range(iters):
        pi_acc = np.zeros(K)
        A_acc = np.zeros((K, K))
        means_acc = np.zeros((K, D))
        vars_acc = np.zeros((K, D))
        gamma_den = np.zeros(K)
        ll = 0.0
        log_pi, log_A = np.log(np.maximum(pi,1e-300)), np.log(np.maximum(A,1e-300))
        gammas_per_seq = []
        for X in seqs:
            log_B = log_diag_gauss_pdf(X, means, vars_)
            logZ, gamma, xi = forward_backward_log(log_pi, log_A, log_B)
            ll += logZ
            gammas_per_seq.append(gamma)
            pi_acc += gamma[0]
            A_acc  += xi.sum(axis=0)
            gamma_den += gamma.sum(axis=0)
            for k in range(K):
                means_acc[k] += (gamma[:,k][:,None] * X).sum(axis=0)
        means = means_acc / np.maximum(gamma_den[:,None], 1e-12)
        for X, gamma in zip(seqs, gammas_per_seq):
            for k in range(K):
                diff = X - means[k]
                vars_acc[k] += (gamma[:,k][:,None] * (diff**2)).sum(axis=0)
        vars_ = vars_acc / np.maximum(gamma_den[:,None], 1e-12)
        vars_ = np.maximum(vars_, var_floor)
        pi = pi_acc / np.maximum(pi_acc.sum(), 1e-12)
        A  = normalize_rows(A_acc + 1e-9)
        if left_right:
            mask = np.zeros_like(A, dtype=bool)
            for i in range(K):
                mask[i,i] = True
                if i < K-1: mask[i,i+1] = True
            A = normalize_rows(A * mask + 1e-12)
        if verbose:
            print(f"Iter {it+1}/{iters}  LL={ll:.3f}")
        if ll - prev_ll < tol:
            break
        prev_ll = ll
    return dict(pi=pi, A=A, means=means, vars_=vars_)

def baum_welch_fast(Xs: List[np.ndarray], K:int=K, iters:int=ITERS, tol:float=TOL,
                    left_right:bool=LEFT_RIGHT, var_floor:float=VAR_FLOOR,
                    use_minibatch:bool=USE_MINIBATCH, batch_fraction:float=BATCH_FRACTION,
                    rng=None, verbose=False):
    rng = rng or np.random.default_rng(0)
    D = Xs[0].shape[1]
    model = init_params(K, D, left_right=left_right, rng=rng)
    if not use_minibatch:
        return baum_welch_diagonal_gaussian(Xs, K, iters, tol, rng, verbose, init=model, var_floor=var_floor, left_right=left_right)
    outer = iters
    for t in range(outer):
        m = max(1, int(np.ceil(len(Xs)*batch_fraction)))
        idx = rng.choice(len(Xs), size=m, replace=False)
        batch = [Xs[i] for i in idx]
        model = baum_welch_diagonal_gaussian(batch, K, iters=2, tol=tol, rng=rng, verbose=False,
                                             init=model, var_floor=var_floor, left_right=left_right)
    return model

def sequence_loglik(model:Dict, X:np.ndarray):
    pi, A, means, vars_ = model['pi'], model['A'], model['means'], model['vars_']
    log_pi, log_A = np.log(np.maximum(pi,1e-300)), np.log(np.maximum(A,1e-300))
    log_B = log_diag_gauss_pdf(X, means, vars_)
    alpha = log_pi + log_B[0]
    for t in range(1, X.shape[0]):
        tmp = alpha[:, None] + log_A
        alpha = log_B[t] + logsumexp_np(tmp, axis=0)
    return float(logsumexp_np(alpha, axis=0))


## 4. Train One HMM per Word

In [5]:
classes = list(range(len(USE_LABELS)))
models = {}
for c in classes:
    Xc = [X[i] for i in range(len(X)) if y[i]==c]
    models[c] = baum_welch_fast(Xc, K=K, iters=ITERS, tol=TOL, left_right=LEFT_RIGHT,
                                var_floor=VAR_FLOOR, use_minibatch=USE_MINIBATCH,
                                batch_fraction=BATCH_FRACTION, rng=rng, verbose=False)
print("Trained models for:", USE_LABELS)


Trained models for: ['yes', 'no', 'up', 'down']


## 5. Toy Classification (Argmax Likelihood)

In [7]:
def predict(models:Dict[int,Dict], Xs:List[np.ndarray]):
    yhat = []
    for X_ in Xs:
        scores = {c: sequence_loglik(models[c], X_) for c in models}
        yhat.append(max(scores, key=scores.get))
    return np.array(yhat)

yhat = predict(models, X)
acc = (yhat == y).mean()
print(f"Train-set accuracy (illustrative): {acc*100:.1f}% over {len(X)} utterances")
for i in range(len(X)):
    print(f"{NAMES[i][0]:>4s} -> pred={USE_LABELS[yhat[i]]}")


Train-set accuracy (illustrative): 60.4% over 48 utterances
 yes -> pred=yes
 yes -> pred=no
 yes -> pred=yes
 yes -> pred=yes
 yes -> pred=yes
 yes -> pred=yes
 yes -> pred=yes
 yes -> pred=yes
 yes -> pred=yes
 yes -> pred=yes
 yes -> pred=yes
 yes -> pred=yes
  no -> pred=no
  no -> pred=up
  no -> pred=no
  no -> pred=no
  no -> pred=no
  no -> pred=down
  no -> pred=no
  no -> pred=no
  no -> pred=down
  no -> pred=yes
  no -> pred=down
  no -> pred=up
  up -> pred=down
  up -> pred=up
  up -> pred=up
  up -> pred=up
  up -> pred=up
  up -> pred=up
  up -> pred=yes
  up -> pred=up
  up -> pred=down
  up -> pred=up
  up -> pred=down
  up -> pred=yes
down -> pred=no
down -> pred=no
down -> pred=up
down -> pred=up
down -> pred=down
down -> pred=down
down -> pred=yes
down -> pred=down
down -> pred=no
down -> pred=down
down -> pred=down
down -> pred=up


## 6. Teaching Knobs (Quick Reference)

- **Faster**: lower `MAX_UTTS_PER_LABEL`, `N_MFCC`, `MAX_FRAMES`; raise `FRAME_SUBSAMPLE`; lower `K`, `ITERS`.
- **More stable**: raise `VAR_FLOOR`.
- **Model structure**: set `LEFT_RIGHT=False` to compare with dense transitions.
- **Minibatch**: set `USE_MINIBATCH=True` and tune `BATCH_FRACTION`.
- **Labels**: adjust `USE_LABELS` to fit your lecture (2–4 words keep training snappy).